# Wprowadzenie do sieci neuronowych i uczenia maszynowego - Sieci Rekurencyjne


---

**Prowadzący:** Piotr Baryczkowski, Jakub Bednarek<br>
**Kontakt:** piotr.baryczkowski@put.poznan.pl<br>

---

## Cel ćwiczeń:
- zapoznanie się z rekurencyjnymi sieciami neuronowymi,
- stworzenie modelu sieci z warstwami rekurencyjnymi dla zbioru danych MNIST,
- stworzenie własnych implementacji warstwami neuronowych

In [1]:
import numpy as np
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.9.1+rocm6.4
CUDA available: True


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [2]:
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [3]:
training_data = datasets.MNIST(
    root="data", train=True, download=True, transform=ToTensor()
)

test_data = datasets.MNIST(
    root="data", train=False, download=True, transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=True)

100.0%

100.0%

100.0%

100.0%



## Sieci rekurencyjne
http://colah.github.io/posts/2015-08-Understanding-LSTMs/

https://pytorch.org/docs/stable/generated/torch.nn.RNN.html

http://karpathy.github.io/2015/05/21/rnn-effectiveness/

http://www.wildml.com/2015/09/recurrent-neural-networks-tutorial-part-1-introduction-to-rnns/

Przykładowy model z warstwą rekurencyjną dla danych MNIST:

In [4]:
class RecurrentModel(nn.Module):
    def __init__(self, num_classes=10):
        super(RecurrentModel, self).__init__()
        self.num_classes = num_classes
        # Define your layers here.
        self.lstm_1 = nn.LSTM(input_size=28, hidden_size=128, batch_first=True)
        self.relu_1 = nn.ReLU()
        self.dense_1 = nn.Linear(128, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, inputs):
        if inputs.dim() == 4:
            # Example: (batch_size, channels, sequence_length, features)
            inputs = inputs.squeeze(1)  # Remove the channels dimension if it's 1
        elif inputs.dim() != 3:
            raise ValueError(f"Expected input to be 3D, got {inputs.dim()}D instead.")

        lstm_out, _ = self.lstm_1(inputs)
        # Take the last output from the sequence (assume inputs are padded appropriately or have consistent lengths)
        x = lstm_out[:, -1, :]  # Get the output of the last time step
        x = self.relu_1(x)
        x = self.dense_1(x)
        return self.softmax(x)


model = RecurrentModel(num_classes=10)
model

RecurrentModel(
  (lstm_1): LSTM(28, 128, batch_first=True)
  (relu_1): ReLU()
  (dense_1): Linear(in_features=128, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [5]:
learning_rate = 1e-3
batch_size = 32
epochs = 5


def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(
        f"Test Error: \n Accuracy: {(100 * correct):>0.1f}%, Avg loss: {test_loss:>8f} \n"
    )

In [6]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.302030  [   32/60000]
loss: 1.993530  [ 3232/60000]
loss: 1.993530  [ 3232/60000]
loss: 1.728767  [ 6432/60000]
loss: 1.728767  [ 6432/60000]
loss: 1.647049  [ 9632/60000]
loss: 1.647049  [ 9632/60000]
loss: 1.672880  [12832/60000]
loss: 1.672880  [12832/60000]
loss: 1.741183  [16032/60000]
loss: 1.741183  [16032/60000]
loss: 1.736418  [19232/60000]
loss: 1.736418  [19232/60000]
loss: 1.694873  [22432/60000]
loss: 1.694873  [22432/60000]
loss: 1.587184  [25632/60000]
loss: 1.587184  [25632/60000]
loss: 1.790498  [28832/60000]
loss: 1.790498  [28832/60000]
loss: 1.575026  [32032/60000]
loss: 1.575026  [32032/60000]
loss: 1.641525  [35232/60000]
loss: 1.641525  [35232/60000]
loss: 1.548756  [38432/60000]
loss: 1.548756  [38432/60000]
loss: 1.536916  [41632/60000]
loss: 1.536916  [41632/60000]
loss: 1.529437  [44832/60000]
loss: 1.529437  [44832/60000]
loss: 1.523683  [48032/60000]
loss: 1.523683  [48032/60000]
loss: 1.504941  [51232/60000]


### Zadanie 1
Rozszerz model z powyższego przykładu o kolejną warstwę rekurencyjną przed gęstą warstwą wyjściową.

Standardowe sieci neuronowe generują jeden wynik na podstawie jednego inputu.
Natomiast sieci rekurencyjne przetwarzają dane sekwencyjnie, w każdym kroku łącząc wynik poprzedniego przetwarzania i aktualnego wejścia. Dlatego domyślnym wejściem sieci neuronowej jest tensor 3-wymiarowy ([batch_size,sequence_size,sample_size]).
Domyślnie warstwy rekurencyjne w PyTorchu zwracają sekwencje wyników wszystkich kroków przetwarzania dla warstwy rekurencyjnej. Jeśli chcesz zwrócić tylko wyniki ostatniego przetwarzania dla warstwy rekurencyjnej, musisz samemu to zaimplementować np. `x = lstm_out[:, -1, :]`.


In [7]:
class RecurrentModel2(nn.Module):
    def __init__(self, num_classes=10):
        super(RecurrentModel2, self).__init__()
        self.num_classes = num_classes
        self.lstm_1 = nn.LSTM(input_size=28, hidden_size=128, batch_first=True)
        self.lstm_2 = nn.LSTM(input_size=128, hidden_size=64, batch_first=True)
        self.relu = nn.ReLU()
        self.dense = nn.Linear(64, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, inputs):
        if inputs.dim() == 4:
            inputs = inputs.squeeze(1)
        elif inputs.dim() != 3:
            raise ValueError(f"Expected input to be 3D, got {inputs.dim()}D instead.")

        lstm_out, _ = self.lstm_1(inputs)
        lstm_out, _ = self.lstm_2(lstm_out)
        x = lstm_out[:, -1, :]
        x = self.relu(x)
        x = self.dense(x)
        return self.softmax(x)


model = RecurrentModel2(num_classes=10)
model

RecurrentModel2(
  (lstm_1): LSTM(28, 128, batch_first=True)
  (lstm_2): LSTM(128, 64, batch_first=True)
  (relu): ReLU()
  (dense): Linear(in_features=64, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [8]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.303727  [   32/60000]
loss: 1.906791  [ 3232/60000]
loss: 1.906791  [ 3232/60000]
loss: 1.861026  [ 6432/60000]
loss: 1.861026  [ 6432/60000]
loss: 1.796464  [ 9632/60000]
loss: 1.796464  [ 9632/60000]
loss: 1.776795  [12832/60000]
loss: 1.776795  [12832/60000]
loss: 1.777121  [16032/60000]
loss: 1.777121  [16032/60000]
loss: 1.629129  [19232/60000]
loss: 1.629129  [19232/60000]
loss: 1.711678  [22432/60000]
loss: 1.711678  [22432/60000]
loss: 1.744279  [25632/60000]
loss: 1.744279  [25632/60000]
loss: 1.798731  [28832/60000]
loss: 1.798731  [28832/60000]
loss: 1.593379  [32032/60000]
loss: 1.593379  [32032/60000]
loss: 1.795560  [35232/60000]
loss: 1.795560  [35232/60000]
loss: 1.647275  [38432/60000]
loss: 1.647275  [38432/60000]
loss: 1.525980  [41632/60000]
loss: 1.525980  [41632/60000]
loss: 1.500993  [44832/60000]
loss: 1.500993  [44832/60000]
loss: 1.580910  [48032/60000]
loss: 1.580910  [48032/60000]
loss: 1.487027  [51232/60000]


### Zadanie 2
Wykorzystując model z przykładu, napisz sieć rekurencyjną przy użyciu RNNCell.

RNNCell implementuje tylko operacje wykonywane przez warstwę
rekurencyjną dla jednego kroku. Warstwy rekurencyjne w każdym kroku
łączą wynik operacji poprzedniego kroku i aktualny input.
Wykorzystaj pętle for do wielokrotnego wywołania komórki RNNCell (liczba kroków to liczba elementów w sekwencji).

Wywołanie zainicjalizowanej komórki rekurencyjnej wymaga podania aktualnego inputu i listy stanów ukrytych poprzedniego kroku (RNNCell ma jeden stan).

Trzeba zainicjalizować ukryty stan warstwy z wartościami początkowymi (można wykorzystać zmienne losowe - torch.rand).

In [9]:
import torch
import torch.nn as nn


class RecurrentModel3(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_classes=10):
        super(RecurrentModel3, self).__init__()
        self.hidden_size = hidden_size
        self.num_classes = num_classes
        self.rnn_cell = nn.RNNCell(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dense = nn.Linear(hidden_size, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, inputs):
        if inputs.dim() == 4:
            inputs = inputs.squeeze(1)
        elif inputs.dim() != 3:
            raise ValueError(f"Expected input to be 3D, got {inputs.dim()}D instead.")

        batch_size = inputs.size(0)
        seq_len = inputs.size(1)

        h = torch.zeros(batch_size, self.hidden_size, device=inputs.device)

        for t in range(seq_len):
            h = self.rnn_cell(inputs[:, t, :], h)

        x = self.relu(h)
        x = self.dense(x)
        return self.softmax(x)


model = RecurrentModel3(input_size=28, hidden_size=128, num_classes=10)

In [10]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.302904  [   32/60000]
loss: 2.312081  [ 3232/60000]
loss: 2.312081  [ 3232/60000]
loss: 2.290031  [ 6432/60000]
loss: 2.290031  [ 6432/60000]
loss: 2.300237  [ 9632/60000]
loss: 2.300237  [ 9632/60000]
loss: 2.301990  [12832/60000]
loss: 2.301990  [12832/60000]
loss: 2.319893  [16032/60000]
loss: 2.319893  [16032/60000]
loss: 2.245117  [19232/60000]
loss: 2.245117  [19232/60000]
loss: 2.301752  [22432/60000]
loss: 2.301752  [22432/60000]
loss: 2.249361  [25632/60000]
loss: 2.249361  [25632/60000]
loss: 2.252379  [28832/60000]
loss: 2.252379  [28832/60000]
loss: 2.228248  [32032/60000]
loss: 2.228248  [32032/60000]
loss: 2.206834  [35232/60000]
loss: 2.206834  [35232/60000]
loss: 2.254341  [38432/60000]
loss: 2.254341  [38432/60000]
loss: 2.168511  [41632/60000]
loss: 2.168511  [41632/60000]
loss: 2.234115  [44832/60000]
loss: 2.234115  [44832/60000]
loss: 2.220855  [48032/60000]
loss: 2.220855  [48032/60000]
loss: 2.122388  [51232/60000]


### Zadanie 3
Zamień komórkę rekurencyjną z poprzedniego zadania na LSTMCell.

In [11]:
class RecurrentModel4(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_classes=10):
        super(RecurrentModel4, self).__init__()
        self.hidden_size = hidden_size
        self.num_classes = num_classes
        self.lstm_cell = nn.LSTMCell(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dense = nn.Linear(hidden_size, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, inputs):
        if inputs.dim() == 4:
            inputs = inputs.squeeze(1)
        elif inputs.dim() != 3:
            raise ValueError(f"Expected input to be 3D, got {inputs.dim()}D instead.")

        batch_size = inputs.size(0)
        seq_len = inputs.size(1)

        h = torch.zeros(batch_size, self.hidden_size, device=inputs.device)
        c = torch.zeros(batch_size, self.hidden_size, device=inputs.device)

        for t in range(seq_len):
            h, c = self.lstm_cell(inputs[:, t, :], (h, c))

        x = self.relu(h)
        x = self.dense(x)
        return self.softmax(x)


model = RecurrentModel4(input_size=28, hidden_size=128, num_classes=10)
model

RecurrentModel4(
  (lstm_cell): LSTMCell(28, 128)
  (relu): ReLU()
  (dense): Linear(in_features=128, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [12]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.304118  [   32/60000]
loss: 2.304229  [ 3232/60000]
loss: 2.304229  [ 3232/60000]
loss: 2.221815  [ 6432/60000]
loss: 2.221815  [ 6432/60000]
loss: 1.838936  [ 9632/60000]
loss: 1.838936  [ 9632/60000]
loss: 1.925356  [12832/60000]
loss: 1.925356  [12832/60000]
loss: 1.774925  [16032/60000]
loss: 1.774925  [16032/60000]
loss: 1.710250  [19232/60000]
loss: 1.710250  [19232/60000]
loss: 1.717745  [22432/60000]
loss: 1.717745  [22432/60000]
loss: 1.560902  [25632/60000]
loss: 1.560902  [25632/60000]
loss: 1.804793  [28832/60000]
loss: 1.804793  [28832/60000]
loss: 1.768798  [32032/60000]
loss: 1.768798  [32032/60000]
loss: 1.681589  [35232/60000]
loss: 1.681589  [35232/60000]
loss: 1.588264  [38432/60000]
loss: 1.588264  [38432/60000]
loss: 1.639180  [41632/60000]
loss: 1.639180  [41632/60000]
loss: 1.526537  [44832/60000]
loss: 1.526537  [44832/60000]
loss: 1.531136  [48032/60000]
loss: 1.531136  [48032/60000]
loss: 1.536900  [51232/60000]


### Zadanie 4
Wykorzystując model z poprzedniego zadania, stwórz model sieci
neuronowej z własną implementacją prostej warstwy rekurencyjnej.
- w call zamień self.lstm_cell_layer(x) na wyołanie własnej metody np. self.cell(x)
- w konstruktorze modelu usuń inicjalizację komórki LSTM i zastąp ją inicjalizacją warstw potrzebnych do stworzenia własnej komórki rekurencyjnej,
- stwórz metodę cell() wykonującą operacje warstwy rekurencyjnej,
- prosta warstwa rekurencyjna konkatenuje poprzedni wyniki i aktualny input, a następnie przepuszcza ten połączony tensor przez warstwę gęstą (Dense).

In [13]:
class RecurrentModel5(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_classes=10):
        super(RecurrentModel5, self).__init__()
        self.hidden_size = hidden_size
        self.num_classes = num_classes
        self.rnn_dense = nn.Linear(input_size + hidden_size, hidden_size)
        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()
        self.dense = nn.Linear(hidden_size, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def cell(self, x, h):
        combined = torch.cat((x, h), dim=1)
        h_new = self.tanh(self.rnn_dense(combined))
        return h_new

    def forward(self, inputs):
        if inputs.dim() == 4:
            inputs = inputs.squeeze(1)
        elif inputs.dim() != 3:
            raise ValueError(f"Expected input to be 3D, got {inputs.dim()}D instead.")

        batch_size = inputs.size(0)
        seq_len = inputs.size(1)

        h = torch.zeros(batch_size, self.hidden_size, device=inputs.device)

        for t in range(seq_len):
            h = self.cell(inputs[:, t, :], h)

        x = self.relu(h)
        x = self.dense(x)
        return self.softmax(x)


model = RecurrentModel5(input_size=28, hidden_size=128, num_classes=10)
model

RecurrentModel5(
  (rnn_dense): Linear(in_features=156, out_features=128, bias=True)
  (tanh): Tanh()
  (relu): ReLU()
  (dense): Linear(in_features=128, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [14]:
epochs = 5

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301688  [   32/60000]
loss: 2.282688  [ 3232/60000]
loss: 2.282688  [ 3232/60000]
loss: 2.287277  [ 6432/60000]
loss: 2.287277  [ 6432/60000]
loss: 2.245496  [ 9632/60000]
loss: 2.245496  [ 9632/60000]
loss: 1.890758  [12832/60000]
loss: 1.890758  [12832/60000]
loss: 1.841529  [16032/60000]
loss: 1.841529  [16032/60000]
loss: 1.877407  [19232/60000]
loss: 1.877407  [19232/60000]
loss: 1.797754  [22432/60000]
loss: 1.797754  [22432/60000]
loss: 1.800104  [25632/60000]
loss: 1.800104  [25632/60000]
loss: 1.942956  [28832/60000]
loss: 1.942956  [28832/60000]
loss: 1.704793  [32032/60000]
loss: 1.704793  [32032/60000]
loss: 1.813298  [35232/60000]
loss: 1.813298  [35232/60000]
loss: 1.842843  [38432/60000]
loss: 1.842843  [38432/60000]
loss: 1.838778  [41632/60000]
loss: 1.838778  [41632/60000]
loss: 1.618610  [44832/60000]
loss: 1.618610  [44832/60000]
loss: 1.754270  [48032/60000]
loss: 1.754270  [48032/60000]
loss: 1.683121  [51232/60000]


### Zadanie 5

Na podstawie modelu z poprzedniego zadania stwórz model z własną implementacją warstwy LSTM. Dokładny i zrozumiały opis działania wartswy LSTM znajduje się na [stronie](http://colah.github.io/posts/2015-08-Understanding-LSTMs/).

In [15]:
from torch.nn.modules.activation import Sigmoid, Tanh


class RecurrentModel6(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_classes=10):
        super(RecurrentModel6, self).__init__()
        self.hidden_size = hidden_size
        self.num_classes = num_classes

        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()
        self.dense = nn.Linear(hidden_size, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def cell(self, x, h, c):
        combined = torch.cat((x, h), dim=1)

        f_t = self.sigmoid(self.W_f(combined))
        i_t = self.sigmoid(self.W_i(combined))
        c_tilde = self.tanh(self.W_c(combined))
        o_t = self.sigmoid(self.W_o(combined))

        c_new = f_t * c + i_t * c_tilde
        h_new = o_t * self.tanh(c_new)

        return h_new, c_new

    def forward(self, inputs):
        if inputs.dim() == 4:
            inputs = inputs.squeeze(1)
        elif inputs.dim() != 3:
            raise ValueError(f"Expected input to be 3D, got {inputs.dim()}D instead.")

        batch_size = inputs.size(0)
        seq_len = inputs.size(1)

        h = torch.zeros(batch_size, self.hidden_size, device=inputs.device)
        c = torch.zeros(batch_size, self.hidden_size, device=inputs.device)

        for t in range(seq_len):
            h, c = self.cell(inputs[:, t, :], h, c)

        x = self.relu(h)
        x = self.dense(x)
        return self.softmax(x)


model = RecurrentModel6(input_size=28, hidden_size=128, num_classes=10)
model

RecurrentModel6(
  (W_f): Linear(in_features=156, out_features=128, bias=True)
  (W_i): Linear(in_features=156, out_features=128, bias=True)
  (W_c): Linear(in_features=156, out_features=128, bias=True)
  (W_o): Linear(in_features=156, out_features=128, bias=True)
  (sigmoid): Sigmoid()
  (tanh): Tanh()
  (relu): ReLU()
  (dense): Linear(in_features=128, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [16]:
epochs = 2
learning_rate = 0.001
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.303607  [   32/60000]
loss: 2.103317  [ 3232/60000]
loss: 2.103317  [ 3232/60000]
loss: 1.952175  [ 6432/60000]
loss: 1.952175  [ 6432/60000]
loss: 1.764335  [ 9632/60000]
loss: 1.764335  [ 9632/60000]
loss: 2.011908  [12832/60000]
loss: 2.011908  [12832/60000]
loss: 1.724005  [16032/60000]
loss: 1.724005  [16032/60000]
loss: 1.799412  [19232/60000]
loss: 1.799412  [19232/60000]
loss: 1.791571  [22432/60000]
loss: 1.791571  [22432/60000]
loss: 1.730096  [25632/60000]
loss: 1.730096  [25632/60000]
loss: 1.705417  [28832/60000]
loss: 1.705417  [28832/60000]
loss: 1.596198  [32032/60000]
loss: 1.596198  [32032/60000]
loss: 1.569980  [35232/60000]
loss: 1.569980  [35232/60000]
loss: 1.647326  [38432/60000]
loss: 1.647326  [38432/60000]
loss: 1.614547  [41632/60000]
loss: 1.614547  [41632/60000]
loss: 1.528752  [44832/60000]
loss: 1.528752  [44832/60000]
loss: 1.522230  [48032/60000]
loss: 1.522230  [48032/60000]
loss: 1.501724  [51232/60000]
